# Sentinel-2 Yearly Snapshots for Espírito Santo

Collect yearly Sentinel-2 scenes covering Espírito Santo (Brazil) to feed the Reflorestar MVP. The notebook targets the AWS copy of the Copernicus Open Access Hub (Sentinel-2 L2A/L1C collections), chooses the clearest scene per year, and stores the requested assets in `../data/sentinel-2/espirito-santo/<year>/`.

> ⚠️ Run the import and download cells manually once the environment has network access. This file only defines the workflow; no data pull was executed.

## Environment preparation
- Enough storage under `../data` for multi-GB Sentinel-2 scenes (one per year starting in 2015). Estimating around 400-500 Mb per Year.

In [9]:
from __future__ import annotations

import datetime as dt
import json
from pathlib import Path
from typing import Dict, Iterable, List, Optional
from urllib.parse import urlparse

import requests

In [13]:
AWS_STAC_URL = "https://earth-search.aws.element84.com/v1"
PREFERRED_COLLECTIONS = ("sentinel-2-l2a", "sentinel-2-l1c")

# Bounding box that tightly covers Espírito Santo, Brazil (lon/lat in WGS84)
AOI_BOUNDING_BOX = [-41.88, -21.30, -39.52, -17.88]

MAX_CLOUD_COVER = 50  # percent; the search logic relaxes this if needed
SEARCH_LIMIT = 100
ASSETS_TO_DOWNLOAD = ["visual", "metadata", "B04", "B08", "SCL"]
CHUNK_SIZE_MB = 8

DATA_ROOT = Path("../data/sentinel-2/espirito-santo")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

current_year = dt.datetime.utcnow().year
TARGET_YEARS = list(range(2015, current_year + 1))

SESSION = requests.Session()
SESSION.headers.update(
    {
        "Accept": "application/geo+json",
        "Content-Type": "application/geo+json",
        "User-Agent": "carbono-data-collection/0-data-gathering",
    }
)

/var/folders/c2/ngdhs83s2kgcfqp8s8m8z84m0000gn/T/ipykernel_27965/2271190123.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_year = dt.datetime.utcnow().year


## Helper functions
The helpers below search the AWS STAC catalog, pick the clearest scene per year (with a graceful fallback for high-cloud periods), download the requested assets, and record the local metadata needed by the MVP pipeline.

In [11]:
def build_search_payload(
    collection: str,
    year: int,
    cloud_threshold: Optional[int],
    limit: int = SEARCH_LIMIT,
) -> Dict:
    """Compose the POST body expected by the STAC Search endpoint."""

    time_range = f"{year}-01-01T00:00:00Z/{year}-12-31T23:59:59Z"
    payload: Dict[str, object] = {
        "collections": [collection],
        "bbox": AOI_BOUNDING_BOX,
        "datetime": time_range,
        "limit": limit,
        "sortby": [{"field": "eo:cloud_cover", "direction": "asc"}],
    }

    if cloud_threshold is not None:
        payload["query"] = {"eo:cloud_cover": {"lt": cloud_threshold}}
    return payload


def search_best_item(
    year: int,
    collections: Iterable[str] | None = None,
    cloud_sequence: Iterable[Optional[int]] | None = None,
) -> Dict:
    """Return the clearest available STAC item for the chosen year."""

    collections = tuple(collections or PREFERRED_COLLECTIONS)
    cloud_sequence = tuple(cloud_sequence or (MAX_CLOUD_COVER, 60, None))
    last_error: Optional[Exception] = None

    for cloud_threshold in cloud_sequence:
        for collection in collections:
            payload = build_search_payload(collection, year, cloud_threshold)
            try:
                response = SESSION.post(
                    f"{AWS_STAC_URL}/search", json=payload, timeout=90
                )
                response.raise_for_status()
            except requests.RequestException as exc:
                last_error = exc
                continue

            features = response.json().get("features", [])
            if not features:
                continue

            best = min(
                features,
                key=lambda feature: feature.get("properties", {}).get("eo:cloud_cover", 101),
            )
            if not str(best.get("id", "")).startswith("S2"):
                continue
            best["collection"] = collection
            best["cloud_threshold"] = cloud_threshold
            return best

    raise ValueError(
        last_error or f"No Sentinel-2 scenes found for {year} and bounding box"
    )


def infer_asset_filename(item_id: str, asset_name: str, asset_href: str) -> str:
    """Create a readable local filename for the requested asset."""

    parsed_path = Path(urlparse(asset_href).path)
    suffix = parsed_path.suffix or ".bin"
    clean_item_id = item_id.replace(" ", "_")
    return f"{clean_item_id}_{asset_name}{suffix}"


def download_asset(asset_href: str, destination: Path) -> Path:
    """Stream an asset from AWS S3 and persist it locally."""

    chunk_size = CHUNK_SIZE_MB * 1024 * 1024
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        return destination

    with SESSION.get(asset_href, stream=True, timeout=120) as response:
        response.raise_for_status()
        with destination.open("wb") as target_file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    target_file.write(chunk)
    return destination


def download_year_assets(
    year: int,
    item: Dict,
    assets: Iterable[str] | None = None,
) -> List[Path]:
    """Download the selected assets for a single STAC item."""

    assets = tuple(assets or ASSETS_TO_DOWNLOAD)
    downloaded_paths: List[Path] = []
    year_dir = DATA_ROOT / str(year)

    for asset_name in assets:
        asset = item.get("assets", {}).get(asset_name)
        if not asset:
            print(f"[{year}] Asset '{asset_name}' missing; skipping")
            continue

        asset_href = asset["href"]
        filename = infer_asset_filename(item["id"], asset_name, asset_href)
        local_path = year_dir / filename
        download_asset(asset_href, local_path)
        downloaded_paths.append(local_path)
    return downloaded_paths


def summarize_item(year: int, item: Dict, downloaded_paths: Iterable[Path]) -> Dict:
    """Capture the metadata needed for repeatable processing."""

    properties = item.get("properties", {})
    return {
        "year": year,
        "product_id": item.get("id"),
        "collection": item.get("collection"),
        "mgrs_tile": properties.get("mgrs:tile"),
        "sensing_time": properties.get("datetime"),
        "cloud_cover": properties.get("eo:cloud_cover"),
        "data_coverage": properties.get("sentinel:data_coverage"),
        "cloud_threshold_applied": item.get("cloud_threshold"),
        "downloaded_assets": [str(path) for path in downloaded_paths],
    }

## 1. Query the STAC catalog for the clearest yearly scene
Execute the cell below to fetch the best-available Sentinel-2 item for every year (2015 through the current one). The search starts with L2A products and automatically falls back to L1C when L2A is not yet available.

In [14]:
yearly_items: Dict[int, Dict] = {}

for year in TARGET_YEARS:
    try:
        item = search_best_item(year)
    except ValueError as exc:
        print(f"[{year}] No usable scene: {exc}")
        continue

    properties = item.get("properties", {})
    yearly_items[year] = item
    print(
        "[{year}] {product} | {collection} | cloud={cloud:.1f}% | coverage={coverage}% | {datetime}".format(
            year=year,
            product=item.get("id"),
            collection=item.get("collection"),
            cloud=properties.get("eo:cloud_cover", float('nan')),
            coverage=properties.get("sentinel:data_coverage"),
            datetime=properties.get("datetime"),
        )
    )

print(f"Collected {len(yearly_items)} scene references out of {len(TARGET_YEARS)} years")

[2015] No usable scene: No Sentinel-2 scenes found for 2015 and bounding box
[2016] No usable scene: No Sentinel-2 scenes found for 2016 and bounding box
[2017] No usable scene: No Sentinel-2 scenes found for 2017 and bounding box
[2018] No usable scene: No Sentinel-2 scenes found for 2018 and bounding box
[2019] No usable scene: No Sentinel-2 scenes found for 2019 and bounding box
[2020] No usable scene: No Sentinel-2 scenes found for 2020 and bounding box
[2021] No usable scene: No Sentinel-2 scenes found for 2021 and bounding box
[2022] No usable scene: No Sentinel-2 scenes found for 2022 and bounding box
[2023] No usable scene: No Sentinel-2 scenes found for 2023 and bounding box
[2024] No usable scene: No Sentinel-2 scenes found for 2024 and bounding box
[2025] No usable scene: No Sentinel-2 scenes found for 2025 and bounding box
Collected 0 scene references out of 11 years


## 2. Download the assets and record a local index
Once the scan looks good, download the requested assets (true-color "visual" GeoTIFF, bands B04/B08, classification layer `SCL`, and the XML metadata) and write an index under `../data/sentinel-2/espirito-santo/sentinel2_es_yearly_index.json`.

In [ ]:
index_entries: List[Dict] = []

for year, item in yearly_items.items():
    downloaded_paths = download_year_assets(year, item)
    index_entries.append(summarize_item(year, item, downloaded_paths))
    print(f"[{year}] Saved {len(downloaded_paths)} assets -> {DATA_ROOT / str(year)}")

index_path = DATA_ROOT / "sentinel2_es_yearly_index.json"
with index_path.open("w", encoding="utf-8") as fp:
    json.dump(index_entries, fp, indent=2)

print(f"Metadata index written to {index_path}")

## Next steps
- Use the per-year directories to feed downstream forest-cover analysis or to build mosaics across seasons if a single scene is not enough.
- Incorporate QA by validating the `SCL` masks against expected forested areas.
- Extend the asset list (e.g., more spectral bands) as the modeling workstream solidifies.